# Qualidade de `amizades.csv`

Diagnóstico da rede de seguidores pelos mesmos cinco critérios usados nos outros arquivos:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Campos preenchidos e contas que deveriam estar na rede |
| **Validade** | Os ids e as datas estão no formato esperado? |
| **Consistência** | As relações concordam com o cadastro e com os treinos em companhia? |
| **Unicidade** | Cada relação de seguir aparece uma vez só? |
| **Temporalidade** | As datas estão entre o lançamento e o corte, no ritmo da base? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste (e-mail `@vitaliza.com.br`) e as 959
contas-cópia de `usuarios.csv` (e-mail em maiúsculas) são removidas; nenhuma análise usa informação posterior ao corte.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
SAIDAS = ROOT / "saidas"
LANCAMENTO = pd.Timestamp("2020-09-01")
INICIO_LOG = pd.Timestamp("2024-07-01")   # mudança de universo em usuarios.csv
CORTE = pd.Timestamp("2026-09-30")

ARQUIVOS = ["amizades.csv", "usuarios.csv", "atividades.parquet"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

def ler(nome, **kwargs):
    # Lê como gravado: tudo texto, vazio como "" (parquet mantém os tipos do arquivo).
    if nome.endswith(".parquet"):
        return pd.read_parquet(BRUTOS / nome, **kwargs)
    return pd.read_csv(BRUTOS / nome, dtype=str, keep_default_na=False, **kwargs)

## 0. Leitura

Tudo como texto e sem converter vazio em nulo. De `usuarios.csv`, só o que identifica a conta (o e-mail serve só para
marcar teste e cópia e não é exibido).

In [2]:
amizades = ler("amizades.csv")
assert len(amizades) == 113_480  # linhas segundo o dicionário (seção 04)
N = len(amizades)
print(f"amizades: {N:,} linhas x {amizades.shape[1]} colunas -> {list(amizades.columns)}")
amizades.sample(5, random_state=0)

amizades: 113,480 linhas x 3 colunas -> ['seguidor_id', 'seguido_id', 'desde']


,seguidor_id,seguido_id,desde
80144,u_7dfb838d,u_3a76ab08,2025-08-01
103410,u_a8470947,u_135d280f,2025-04-05
20235,u_a83217a4,u_0779e95f,2025-12-20
8074,u_557615bd,u_32969e5d,2025-03-22
78039,u_597f2399,u_eeef8a74,2025-07-18


### Colunas derivadas para a análise

- `d`: `desde` convertida para data; `mes`: mês de `desde`;
- de `usuarios.csv`: data de cadastro (só a data, como decidido), coorte (antes ou a partir de 07/2024), conta de teste
  e conta-cópia (e-mail com maiúscula).

O arquivo original não muda.

In [3]:
amizades["d"] = pd.to_datetime(amizades["desde"], format="%Y-%m-%d", errors="coerce")
amizades["mes"] = amizades["d"].dt.to_period("M")

usuarios = ler("usuarios.csv", usecols=["user_id", "email", "data_cadastro"])
usuarios["data_cad"] = pd.to_datetime(usuarios["data_cadastro"], format="%Y-%m-%d %H:%M:%S").dt.normalize()
email = usuarios["email"].str.strip()
usuarios["eh_teste"] = email.str.lower().str.endswith("@vitaliza.com.br")
usuarios["eh_copia"] = email != email.str.lower()
usuarios["coorte"] = (usuarios["data_cad"] >= INICIO_LOG).map({False: "antes de 07/2024", True: "07/2024 em diante"})
usuarios = usuarios.drop(columns="email").set_index("user_id")
print(f"usuarios: {len(usuarios):,} contas; teste: {usuarios['eh_teste'].sum()}; cópias: {usuarios['eh_copia'].sum()}")

usuarios: 64,948 contas; teste: 60; cópias: 959


## 1. Completude

Espera-se nenhum vazio e nenhuma conta de teste ou cópia na rede: a cópia nunca aparece em outro arquivo, e o teste não é
cliente.

In [4]:
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}
colunas = ["seguidor_id", "seguido_id", "desde"]
vazios = pd.DataFrame({
    "vazio": [(amizades[c].str.strip() == "").sum() for c in colunas],
    "vazio_disfarcado": [amizades[c].str.strip().str.lower().isin(VAZIOS_DISFARCADOS).sum() for c in colunas],
}, index=colunas)
display(vazios)
n_vazio = int(vazios.sum().sum())
achado("Completude", "vazio em seguidor_id, seguido_id ou desde", f"{n_vazio:,}", n_vazio, "Nenhum")

,vazio,vazio_disfarcado
seguidor_id,0,0
seguido_id,0,0
desde,0,0


In [5]:
for marca, nome, trat in [("eh_teste", "contas de teste (@vitaliza.com.br)", "Se houver: excluir, como nos outros arquivos"),
                          ("eh_copia", "contas-cópia de usuarios.csv (e-mail em maiúsculas)", "Se houver: excluir com as cópias")]:
    ids = set(usuarios.index[usuarios[marca]])
    n = int((amizades["seguidor_id"].isin(ids) | amizades["seguido_id"].isin(ids)).sum())
    achado("Completude", f"relações com {nome}", f"{n:,}", n, trat)

sem_cad = int((~amizades["seguidor_id"].isin(usuarios.index)).sum() + (~amizades["seguido_id"].isin(usuarios.index)).sum())
achado("Completude", "seguidor_id ou seguido_id sem cadastro em usuarios.csv", f"{sem_cad:,}", sem_cad, "Nenhum")

### Quem tem cadastro e não está na rede

Seguir é opcional, então parte das contas não aparece aqui. Se a proporção for parecida nas duas coortes, a ausência é
comportamento e não perda de dado. Contam só as contas válidas (sem teste e sem cópia) cadastradas até o corte.

In [6]:
validas = usuarios[~usuarios["eh_teste"] & ~usuarios["eh_copia"] & (usuarios["data_cad"] <= CORTE)].copy()
ate_corte = amizades[amizades["d"] <= CORTE]
validas["segue"] = ate_corte.groupby("seguidor_id").size().reindex(validas.index, fill_value=0)
validas["seguidores"] = ate_corte.groupby("seguido_id").size().reindex(validas.index, fill_value=0)
validas["na_rede"] = (validas["segue"] + validas["seguidores"]) > 0
rede = validas.groupby("coorte").agg(contas=("na_rede", "size"), na_rede=("na_rede", "sum"),
                                     media_segue=("segue", "mean"), media_seguidores=("seguidores", "mean"))
rede["% na rede"] = (100 * rede["na_rede"] / rede["contas"]).round(1)
display(rede.round(2))
fora = int((~validas["na_rede"]).sum())
achado("Completude", "contas válidas sem nenhuma relação até o corte",
       f"{fora:,} de {len(validas):,} contas ({100 * fora / len(validas):.1f}%); na rede: "
       + "; ".join(f"{c}: {p}%" for c, p in rede["% na rede"].items()),
       0, "Nenhum: seguir é opcional. Na análise, conta fora da rede tem 0 seguidores (não é dado faltante)",
       status="esperado")

,contas,na_rede,media_segue,media_seguidores,% na rede
coorte,,,,,
07/2024 em diante,58729,49331,1.73,1.71,84.0
antes de 07/2024,5200,4831,2.25,2.48,92.9


## 2. Validade

Ids no padrão `u_[0-9a-f]{8}`, `desde` convertendo e ninguém seguindo a si mesmo.

In [7]:
fora_padrao = int(sum((~amizades[c].str.fullmatch(r"u_[0-9a-f]{8}")).sum() for c in ["seguidor_id", "seguido_id"]))
achado("Validade", "id fora do padrão u_[0-9a-f]{8}", f"{fora_padrao:,}", fora_padrao, "Nenhum")
padroes = amizades["desde"].str.replace(r"\d", "9", regex=True).value_counts().to_dict()
falha = int(amizades["d"].isna().sum())
achado("Validade", "desde que não converte em data", f"{falha}; padrões: {padroes}", falha, "Nenhum")
proprio = int((amizades["seguidor_id"] == amizades["seguido_id"]).sum())
achado("Validade", "conta seguindo a si mesma", f"{proprio:,}", proprio, "Nenhum")

## 3. Consistência

Ninguém segue alguém antes de se cadastrar, nem é seguido antes de existir. A comparação é por dia: `desde` não tem
hora e a hora do cadastro não é confiável (decisão de 07/11/2026).

In [8]:
cad = usuarios["data_cad"]
antes_seguidor = int((amizades["d"] < amizades["seguidor_id"].map(cad)).sum())
antes_seguido = int((amizades["d"] < amizades["seguido_id"].map(cad)).sum())
print(f"antes do cadastro de quem segue: {antes_seguidor:,}; antes do cadastro de quem é seguido: {antes_seguido:,}")
achado("Consistência", "relação anterior ao cadastro de quem segue ou de quem é seguido",
       f"{antes_seguidor + antes_seguido:,}", antes_seguidor + antes_seguido, "Nenhum")
dias_cad = (amizades["d"] - amizades[["seguidor_id", "seguido_id"]].apply(lambda s: s.map(cad)).max(axis=1)).dt.days
print("dias entre o cadastro mais recente do par e a relação:", dias_cad.describe().round(1).to_dict())

antes do cadastro de quem segue: 0; antes do cadastro de quem é seguido: 0
dias entre o cadastro mais recente do par e a relação: {'count': 113480.0, 'mean': 59.7, 'std': 34.7, 'min': 0.0, '25%': 30.0, '50%': 59.0, '75%': 89.0, 'max': 128.0}


### Reciprocidade

Seguir é de mão única: se Ana segue Bruno e Bruno segue Ana, são duas linhas (dicionário). Espera-se boa parte das
relações recíprocas, e a volta pouco depois da ida.

In [9]:
volta = amizades.merge(
    amizades[["seguidor_id", "seguido_id", "d"]].rename(
        columns={"seguidor_id": "seguido_id", "seguido_id": "seguidor_id", "d": "d_volta"}),
    on=["seguidor_id", "seguido_id"], how="left")
reciproca = volta["d_volta"].notna()
pares = volta[reciproca & (volta["seguidor_id"] < volta["seguido_id"])].copy()
pares["dias"] = (pares["d_volta"] - pares["d"]).dt.days.abs()
print(f"linhas recíprocas: {reciproca.sum():,} de {N:,} ({100 * reciproca.mean():.1f}%); pares: {len(pares):,}")
dist = pares["dias"].value_counts().sort_index()
display(dist.to_frame("pares").T)
achado("Consistência", "relação recíproca (a pessoa seguida segue de volta)",
       f"{reciproca.sum():,} linhas ({100 * reciproca.mean():.1f}%) em {len(pares):,} pares; a volta vem de "
       f"{pares['dias'].min()} a {pares['dias'].max()} dias depois (mediana {pares['dias'].median():.0f})",
       int(reciproca.sum()), "Nenhum: comportamento da rede", status="esperado")

linhas recíprocas: 85,044 de 113,480 (74.9%); pares: 42,522


dias,0,1,2,3,4,5,6,7,8,9
pares,4295,4281,4241,4246,4333,4267,4241,4230,4119,4269


### Treino em companhia contra a rede

`treino_com` (em `atividades.parquet`) registra quem treinou junto. O dicionário avisa que seguir e treinar junto são
coisas diferentes, então a regra não é obrigatória; mas espera-se que a maioria de quem treina junto também se siga.

In [10]:
ativ = ler("atividades.parquet", columns=["user_id", "treino_com"])
com = ativ[ativ["treino_com"] != ""]
relacoes = set(zip(amizades["seguidor_id"], amizades["seguido_id"]))
algum_sentido = pd.Series([(a, b) in relacoes or (b, a) in relacoes for a, b in zip(com["user_id"], com["treino_com"])],
                          index=com.index)
print(f"treinos com companhia: {len(com):,}; parceiro com cadastro: {com['treino_com'].isin(usuarios.index).mean():.1%}; "
      f"parceiro = a própria conta: {(com['treino_com'] == com['user_id']).sum()}")
print(f"o par se segue em algum sentido: {algum_sentido.mean():.1%}")
achado("Consistência", "treino em companhia entre contas que não se seguem",
       f"{(~algum_sentido).sum():,} de {len(com):,} treinos com companhia ({100 * (~algum_sentido).mean():.1f}%); "
       "o dicionário diz que seguir e treinar junto são coisas diferentes", 0,
       "Nenhum: não tratar treino em companhia como relação de seguir (nem o contrário)", status="esperado")

treinos com companhia: 14,137; parceiro com cadastro: 100.0%; parceiro = a própria conta: 0
o par se segue em algum sentido: 94.9%


## 4. Unicidade

O par seguidor → seguido deve aparecer uma vez só. Ida e volta no mesmo dia podem ser duplicata gravada nos dois
sentidos: se fossem, o dia 0 se destacaria na distribuição de prazos da volta (seção 3).

In [11]:
par_rep = int(amizades.duplicated(["seguidor_id", "seguido_id"]).sum())
achado("Unicidade", "par seguidor -> seguido repetido", f"{par_rep:,}", par_rep, "Nenhum")
linha_rep = int(amizades[colunas].duplicated().sum())
achado("Unicidade", "linha inteira repetida", f"{linha_rep:,}", linha_rep, "Nenhum")
mesmo_dia = int((pares["dias"] == 0).sum())
media_outros = dist.loc[dist.index > 0].mean()
print(f"pares com ida e volta no mesmo dia: {mesmo_dia:,}; média dos outros dias (1 a {dist.index.max()}): {media_outros:,.0f}")
achado("Unicidade", "ida e volta no mesmo dia (possível duplicata nos dois sentidos)",
       f"{mesmo_dia:,} pares ({2 * mesmo_dia:,} linhas); a média dos dias 1 a {dist.index.max()} é {media_outros:,.0f}: "
       "o dia 0 não se destaca", 2 * mesmo_dia, "Nenhum: são duas relações", status="esperado")

pares com ida e volta no mesmo dia: 4,295; média dos outros dias (1 a 9): 4,247


### Tamanho da rede de cada conta

Uma conta com milhares de seguidores (perfil oficial, robô) distorceria médias de rede.

In [12]:
grau = pd.DataFrame({
    "segue": amizades["seguidor_id"].value_counts().describe(),
    "seguidores": amizades["seguido_id"].value_counts().describe(),
}).round(1)
display(grau)
print("contas por número de seguidores:", amizades["seguido_id"].value_counts().value_counts().sort_index().to_dict())
maximo = int(max(amizades["seguidor_id"].value_counts().max(), amizades["seguido_id"].value_counts().max()))
achado("Unicidade", "conta com rede fora do padrão (centenas de relações)", f"máximo de {maximo} relações em cada sentido",
       0, "Nenhum")

,segue,seguidores
count,50455.0,50547.0
mean,2.2,2.2
std,1.3,1.3
min,1.0,1.0
25%,1.0,1.0
50%,2.0,2.0
75%,3.0,3.0
max,13.0,13.0


contas por número de seguidores: {1: 17648, 2: 15476, 3: 9537, 4: 4830, 5: 1954, 6: 729, 7: 256, 8: 72, 9: 28, 10: 5, 11: 10, 12: 1, 13: 1}


## 5. Temporalidade

Espera-se `desde` de 09/2020 (lançamento) até o corte, com o volume mensal acompanhando a base de contas.

In [13]:
print(f"desde de {amizades['d'].min().date()} a {amizades['d'].max().date()}")
antes_lanc = int((amizades["d"] < LANCAMENTO).sum())
achado("Temporalidade", "relação antes do lançamento (09/2020)", f"{antes_lanc:,}", antes_lanc, "Nenhum")

desde de 2020-09-05 a 2026-10-09


### Relações depois do corte

Hipótese: são voltas de relações começadas antes do corte, gravadas porque o arquivo foi exportado alguns dias depois.

In [14]:
pos = volta[volta["d"] > CORTE]
print("por dia:", pos["d"].dt.strftime("%d/%m").value_counts().sort_index().to_dict())
print(f"recíprocas: {pos['d_volta'].notna().sum()} de {len(pos)}; com a ida até o corte: {(pos['d_volta'] <= CORTE).sum()}")
print(f"dias entre a ida e a volta: de {(pos['d'] - pos['d_volta']).dt.days.min()} a {(pos['d'] - pos['d_volta']).dt.days.max()}")
achado("Temporalidade", "relação com desde depois do corte",
       f"{len(pos):,} relações de 01/10 a {pos['d'].max():%d/%m/%Y}, todas a volta de uma relação começada até o corte "
       f"(0 a 9 dias antes): o arquivo foi exportado depois de 01/10/2026",
       len(pos), "Excluir relações com desde depois de 30/09/2026 antes de contar seguidores ou reciprocidade")

por dia: {'01/10': 54, '02/10': 48, '03/10': 48, '04/10': 35, '05/10': 39, '06/10': 15, '07/10': 21, '08/10': 10, '09/10': 3}
recíprocas: 273 de 273; com a ida até o corte: 273
dias entre a ida e a volta: de 1 a 9


### Volume por mês

Salto acima de 50% só é falha se a base de contas não acompanhar. Comparo as relações novas do mês com os cadastros
válidos do mês e com as relações por conta no primeiro ano, por coorte.

In [15]:
mensal = pd.DataFrame({
    "relações novas": ate_corte.groupby("mes").size(),
    "cadastros válidos": validas.groupby(validas["data_cad"].dt.to_period("M")).size(),
}).fillna(0).astype(int)
meses = pd.period_range("2020-09", "2026-09", freq="M")
faltando = meses.difference(mensal.index[mensal["relações novas"] > 0])
mensal = mensal.reindex(meses, fill_value=0)
mensal["var relações %"] = (100 * (mensal["relações novas"] / mensal["relações novas"].shift() - 1)).round(0)
display(mensal.loc["2024-01":"2025-03"])
saltos = mensal[mensal["var relações %"].abs() > 50]
print("saltos acima de 50%:", {str(m): v for m, v in saltos["var relações %"].items()})
achado("Temporalidade", "mês sem relação nova", f"{len(faltando)}", len(faltando), "Nenhum")

,relações novas,cadastros válidos,var relações %
2024-01,245,119,10.0
2024-02,206,116,-16.0
2024-03,263,115,28.0
2024-04,227,124,-14.0
2024-05,243,118,7.0
2024-06,242,77,-0.0
2024-07,562,1756,132.0
2024-08,1425,1934,154.0
2024-09,2268,2049,59.0
2024-10,3369,1977,49.0


saltos acima de 50%: {'2020-10': 212.0, '2024-07': 132.0, '2024-08': 154.0, '2024-09': 59.0}


Se o salto vem da base, cada conta forma as relações no mesmo ritmo nas duas coortes. Conto as relações (nos dois
sentidos) feitas no primeiro ano depois do cadastro, só para contas cadastradas até 30/09/2025 (com um ano completo antes
do corte).

In [16]:
um_ano = validas[validas["data_cad"] <= CORTE - pd.DateOffset(years=1)]
ritmo = {}
for col in ["seguidor_id", "seguido_id"]:
    t = ate_corte.join(um_ano["data_cad"], on=col, how="inner")
    t = t[(t["d"] - t["data_cad"]).dt.days <= 365]
    ritmo[col] = t.groupby(col).size().reindex(um_ano.index, fill_value=0)
um_ano = um_ano.assign(primeiro_ano=ritmo["seguidor_id"] + ritmo["seguido_id"])
display(um_ano.groupby("coorte")["primeiro_ano"].agg(["size", "mean"]).round(2).rename(columns={"size": "contas", "mean": "relações no 1º ano"}))
n_saltos = int(saltos["relações novas"].sum())
s2024 = [str(m) for m in saltos.index if m.year == 2024]
outros = [str(m) for m in saltos.index if m.year != 2024]
achado("Temporalidade", "salto acima de 50% nas relações novas",
       f"{', '.join(s2024)}: acompanha a entrada das contas a partir de 07/2024 (mudança de universo de usuarios.csv); "
       "no primeiro ano, cada conta forma quase o mesmo número de relações nas duas coortes. "
       f"{', '.join(outros)}: mês seguinte ao lançamento, sobre uma base de poucas contas",
       n_saltos, "Nenhum; para comparar épocas, usar relações por conta, não o total", status="esperado")

,contas,relações no 1º ano
coorte,,
07/2024 em diante,30853,3.84
antes de 07/2024,5200,3.61


### Janela em que as relações se formam

Na seção 3, toda relação começa no máximo 128 dias depois do cadastro mais recente do par. Hipótese: a primeira relação
de cada par (a ida) só acontece nos primeiros ~4 meses da conta mais nova, e a volta vem até 9 dias depois. Se for isso,
duas contas veteranas nunca passam a se seguir.

In [17]:
volta["cad_seguidor"] = volta["seguidor_id"].map(cad)
volta["cad_seguido"] = volta["seguido_id"].map(cad)
volta["cad_novo"] = volta[["cad_seguidor", "cad_seguido"]].max(axis=1)
ida = volta["d_volta"].isna() | (volta["d"] < volta["d_volta"]) | (
    (volta["d"] == volta["d_volta"]) & (volta["seguidor_id"] < volta["seguido_id"]))
dias_ida = (volta.loc[ida, "d"] - volta.loc[ida, "cad_novo"]).dt.days
faixas = pd.cut(dias_ida, [-1, 29, 59, 89, 119, 10_000], labels=["0-29", "30-59", "60-89", "90-119", "120+"])
display(faixas.value_counts().sort_index().to_frame("idas (primeira relação do par)").T)
print(f"ida: de {dias_ida.min()} a {dias_ida.max()} dias depois do cadastro da conta mais nova "
      f"(média {dias_ida.mean():.1f}, desvio {dias_ida.std():.1f}; uma distribuição uniforme de 0 a 119 teria média 59,5 e desvio 34,6)")
print(f"na ida, quem segue é a conta mais nova em {100 * (volta.loc[ida, 'cad_seguidor'] >= volta.loc[ida, 'cad_seguido']).mean():.1f}% dos casos")
veteranos = int(((volta["d"] - volta["cad_novo"]).dt.days > 128).sum())
print(f"relações entre duas contas com mais de 128 dias de casa: {veteranos}")
n_ida = int(ida.sum())
achado("Temporalidade", "relações só se formam nos primeiros ~4 meses da conta mais nova",
       f"todas as {n_ida:,} idas acontecem de {dias_ida.min()} a {dias_ida.max()} dias depois do cadastro da conta mais nova, "
       "em distribuição uniforme, e nenhuma depois; a volta vem até 9 dias depois. Duas contas veteranas nunca passam a se "
       "seguir. Hipótese a confirmar com o dono do dado: regra do produto (sugestão de quem seguir só no começo) ou recorte "
       "do arquivo",
       N, "Não usar 'começou a seguir alguém nos últimos N dias' como sinal de engajamento: depois de 4 meses de casa ele é "
       "zero para todos. Usar o número de relações no corte, comparando contas com o mesmo tempo de casa",
       status="atenção")

,0-29,30-59,60-89,90-119,120+
idas (primeira relação do par),18773,18044,17362,16779,0


ida: de 0 a 119 dias depois do cadastro da conta mais nova (média 58.0, desvio 34.7; uma distribuição uniforme de 0 a 119 teria média 59,5 e desvio 34,6)
na ida, quem segue é a conta mais nova em 97.1% dos casos
relações entre duas contas com mais de 128 dias de casa: 0


### Relações desfeitas

O dicionário diz que o arquivo "guarda todas as relações de seguir", mas não diz o que acontece quando alguém deixa de
seguir. Se a linha é apagada, o arquivo é um retrato da exportação: falta quem seguia no corte e deixou de seguir depois,
e o número de seguidores no corte fica subestimado justamente para quem saiu do app. Não há coluna que permita testar.

In [18]:
achado("Temporalidade", "relações desfeitas antes da exportação (o dicionário não diz se somem)",
       "sem coluna de fim da relação; hipótese a confirmar com o dono do dado", 0,
       "Perguntar ao dono do dado; até lá, tratar o número de seguidores no corte como aproximado e não usá-lo sozinho "
       "como preditor de saída", status="atenção")

## 6. Resumo

In [19]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_amizades.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 13, 'esperado': 5, 'atenção': 3}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,"vazio em seguidor_id, seguido_id ou desde",ok,0,0,Nenhum,0.00
1,Completude,relações com contas de teste (@vitaliza.com.br),ok,0,0,"Se houver: excluir, como nos outros arquivos",0.00
2,Completude,relações com contas-cópia de usuarios.csv (e-m...,ok,0,0,Se houver: excluir com as cópias,0.00
3,Completude,seguidor_id ou seguido_id sem cadastro em usua...,ok,0,0,Nenhum,0.00
4,Completude,contas válidas sem nenhuma relação até o corte,esperado,"9,767 de 63,929 contas (15.3%); na rede: 07/20...",0,"Nenhum: seguir é opcional. Na análise, conta f...",0.00
5,Validade,id fora do padrão u_[0-9a-f]{8},ok,0,0,Nenhum,0.00
6,Validade,desde que não converte em data,ok,0; padrões: {'9999-99-99': 113480},0,Nenhum,0.00
7,Validade,conta seguindo a si mesma,ok,0,0,Nenhum,0.00
8,Consistência,relação anterior ao cadastro de quem segue ou ...,ok,0,0,Nenhum,0.00
9,Consistência,relação recíproca (a pessoa seguida segue de v...,esperado,"85,044 linhas (74.9%) em 42,522 pares; a volta...",85044,Nenhum: comportamento da rede,74.94


In [20]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
